# Initialization checks

Small CPU examples using our adapter code. No downloads needed.

In [ ]:
import pandas as pd
import torch

from dora_spectral import ControlledLinear
from dora_spectral.initialization import decompose, initialize_factors

generator = torch.Generator().manual_seed(42)
weight = torch.randn(5, 9, generator=generator, dtype=torch.float64)
inputs = torch.randn(4, 9, generator=generator, dtype=torch.float64)
svd = decompose(weight, rank=3)

## Matching spectra

Spectral and random factors have the same singular values. Each subtracts its own initial product so both start at the original weight.

In [ ]:
b_s, a_s = initialize_factors(svd, "spectral", gamma=0.5, seed=0)
b_r, a_r = initialize_factors(svd, "random", gamma=0.5, seed=0)
spectra = pd.DataFrame(
    {
        "spectral product": torch.linalg.svdvals(b_s @ a_s).numpy(),
        "random product": torch.linalg.svdvals(b_r @ a_r).numpy(),
    }
)
assert torch.allclose(torch.linalg.svdvals(b_s @ a_s), torch.linalg.svdvals(b_r @ a_r))
spectra

In [ ]:
rows = []
for rule in ("additive", "detached", "exact"):
    for initialization in ("spectral", "random"):
        layer = ControlledLinear(
            weight, rank=3, gamma=0.5, rule=rule, initialization=initialization, decomposition=svd
        )
        error = (layer(inputs) - inputs @ weight.T).abs().max().item()
        assert error < 1e-10
        rows.append(
            {"rule": rule, "initialization": initialization, "initial output max error": error}
        )
pd.DataFrame(rows)

## DoRA gradients

Exact and detached DoRA give the same output here, but different factor gradients.

In [ ]:
exact = ControlledLinear(weight, rank=3, rule="exact")
detached = ControlledLinear(weight, rank=3, rule="detached")
assert torch.allclose(exact(inputs), detached(inputs))
exact(inputs).square().sum().backward()
detached(inputs).square().sum().backward()
assert not torch.allclose(exact.B.grad, detached.B.grad)
assert torch.allclose(exact.magnitude.grad, detached.magnitude.grad)
pd.Series(
    {
        "exact B gradient norm": exact.B.grad.norm().item(),
        "detached B gradient norm": detached.B.grad.norm().item(),
        "gradient difference norm": (exact.B.grad - detached.B.grad).norm().item(),
    }
)